**PIPELINE PRINCIPAL — Stress Test de Liquidité Bancaire**


Auteur    : Stress Test Liquidité


Description : Orchestre l'exécution complète du pipeline :

              Données → ML → Scénarios → Monte-Carlo → Analyse avancée → Rapport



Usage :


    python main.py


Sorties générées dans /tmp/stress_test_output/ :
    - 01_data_overview.png
    - 02_ml_results.png
    - 03_scenarios.png
    - 04_monte_carlo.png
    - 05_advanced_analysis.png
    - risk_measures.csv
    - model_validation.json
    - rapport_risques.txt

In [10]:
import os
import sys
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings('ignore')
sys.path.insert(0, 'C:/Users/X1 Carbon/Desktop/Stress Testing Projet/stress_test_liquidite')

# Répertoire de sortie
OUTPUT_DIR = 'C:/Users/X1 Carbon/Desktop/Stress Testing Projet'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 70)
print("  STRESS TEST DE LIQUIDITÉ BANCAIRE")
print("  Projet complet — Économie Quantitative & Risk Management")
print("=" * 70)


  STRESS TEST DE LIQUIDITÉ BANCAIRE
  Projet complet — Économie Quantitative & Risk Management


In [21]:
# =============================================================================
# ÉTAPE 1 : DONNÉES
# =============================================================================
print("\n[1/6] Génération de la base de données bancaire...")
t0 = time.time()

from simulate_data import create_full_dataset
df = create_full_dataset()
df.to_csv(f'{OUTPUT_DIR}/data_bancaire.csv')


[1/6] Génération de la base de données bancaire...
Dataset créé : 57 observations × 32 variables
Période : 2019-04-30 → 2023-12-31

Statistiques descriptives (variables clés) :
       pib_growth  inflation  depots_vue    hqla  flux_nets_30j  ratio_stress
count       57.00      57.00       57.00   57.00          57.00         57.00
mean         4.70       2.40      178.45  149.81          29.48          0.20
std          1.35       0.91       14.63   12.73           1.78          0.02
min          1.44       0.70      154.21  123.64          25.68          0.16
25%          3.87       1.90      168.32  136.68          28.17          0.19
50%          4.65       2.26      173.38  155.04          29.44          0.20
75%          5.38       2.86      193.52  159.48          30.71          0.21
max          8.39       5.07      201.67  167.65          33.06          0.23


In [22]:
# Graphique aperçu des données
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
fig.suptitle('Aperçu des données bancaires simulées', fontsize=14, fontweight='bold')

plots = [
    ('pib_growth', 'Croissance PIB (%/an)', '#1a9641'),
    ('inflation',  'Inflation (%)',          '#d7191c'),
    ('depots_vue', 'Dépôts à vue (Mds XOF)', '#2c7bb6'),
    ('hqla',       'HQLA (Mds XOF)',          '#7b3294'),
    ('flux_nets_30j', 'Flux nets 30j (Mds)', '#f46d43'),
    ('ratio_transformation', 'Ratio transformation', '#fdae61'),
]

for ax, (col, title, color) in zip(axes.flatten(), plots):
    if col in df.columns:
        ax.plot(df.index, df[col], color=color, linewidth=2)
        ax.fill_between(df.index, df[col].min(), df[col], alpha=0.15, color=color)
        ax.set_title(title, fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.set_xlabel('')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/01_data_overview.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"  ✓ Données générées ({len(df)} obs × {len(df.columns)} variables)")
print(f"  ✓ Temps : {time.time()-t0:.1f}s")

  ✓ Données générées (57 obs × 32 variables)
  ✓ Temps : 12.5s


In [23]:
# =============================================================================
# ÉTAPE 2 : MACHINE LEARNING
# =============================================================================
print("\n[2/6] Entraînement des modèles Machine Learning...")
t0 = time.time()

from ml_models import (prepare_features, evaluate_models,
                               train_best_model, compute_feature_importance,
                               plot_model_results)

X, y, feature_names = prepare_features(df)
results_df = evaluate_models(X, y, feature_names, n_splits=4)
model, scaler, model_name = train_best_model(X, y)
importance_df = compute_feature_importance(model, X, y, feature_names, scaler)

fig_ml = plot_model_results(df, model, scaler, feature_names,
                             importance_df, results_df,
                             save_path=f'{OUTPUT_DIR}/02_ml_results.png')
plt.close()
print(f"  ✓ Meilleur modèle : {model_name}")
print(f"  ✓ Temps : {time.time()-t0:.1f}s")

# =============================================================================
# ÉTAPE 3 : SCÉNARIOS DE STRESS
# =============================================================================
print("\n[3/6] Construction des scénarios de stress...")
t0 = time.time()

from stress_scenarios import (define_scenarios, apply_all_scenarios,
                                         print_scenario_summary, plot_scenarios)

scenarios = define_scenarios()
summary_df = print_scenario_summary()
scenario_results_df = apply_all_scenarios(df)
scenario_results_df.to_csv(f'{OUTPUT_DIR}/scenario_results.csv', index=False)

fig_scen = plot_scenarios(save_path=f'{OUTPUT_DIR}/03_scenarios.png')
plt.close()
print(f"  ✓ {len(scenarios)} scénarios définis")
print(f"  ✓ Temps : {time.time()-t0:.1f}s")


[2/6] Entraînement des modèles Machine Learning...
ÉVALUATION DES MODÈLES — WALK-FORWARD CROSS-VALIDATION
  Ridge Regression          RMSE=2.740  MAE=2.274  R²=-3.4543
  Random Forest             RMSE=1.682  MAE=1.309  R²=-0.3146
  Gradient Boosting         RMSE=1.792  MAE=1.376  R²=-0.4863
  XGBoost                   RMSE=1.764  MAE=1.421  R²=-0.4426
  Réseau de neurones        RMSE=9.597  MAE=7.496  R²=-51.4141

CLASSEMENT PAR RMSE (meilleur modèle en premier) :
            Modèle  RMSE (Mds)  MAE (Mds)  MAPE (%)       R²
     Random Forest       1.682      1.309      4.36  -0.3146
           XGBoost       1.764      1.421      4.77  -0.4426
 Gradient Boosting       1.792      1.376      4.61  -0.4863
  Ridge Regression       2.740      2.274      7.77  -3.4543
Réseau de neurones       9.597      7.496     25.28 -51.4141

Modèle final entraîné : XGBoost
  R² (train) = 0.9994
  RMSE (train) = 0.042 Mds XOF

TOP 10 VARIABLES LES PLUS IMPORTANTES :
                  Feature  Importance

In [16]:
# =============================================================================
# ÉTAPE 4 : SIMULATION MONTE-CARLO
# =============================================================================
print("\n[4/6] Simulation Monte-Carlo (10 000 scénarios × 4 stress)...")
t0 = time.time()

from monte_carlo import (run_monte_carlo, compute_risk_measures,
                                     plot_monte_carlo, interpret_results)

mc_results = run_monte_carlo(
    df=df,
    model=model,
    scaler=scaler,
    feature_names=feature_names,
    scenarios=scenarios,
    n_simulations=10_000,
    use_student_t=True,
    df_t=5.0,
)

risk_measures = compute_risk_measures(mc_results)
risk_measures.to_csv(f'{OUTPUT_DIR}/risk_measures.csv', index=False)

rapport_texte = interpret_results(risk_measures)
with open(f'{OUTPUT_DIR}/rapport_risques.txt', 'w', encoding='utf-8') as f:
    f.write(rapport_texte)

fig_mc = plot_monte_carlo(mc_results, risk_measures,
                           save_path=f'{OUTPUT_DIR}/04_monte_carlo.png')
plt.close()
print(f"  ✓ {10_000 * len(scenarios):,} simulations effectuées")
print(f"  ✓ Temps : {time.time()-t0:.1f}s")


[4/6] Simulation Monte-Carlo (10 000 scénarios × 4 stress)...
TEST DE NORMALITÉ (Jarque-Bera) :
  pib_growth           p=0.5599  normale ✓
  inflation            p=0.7302  normale ✓
  chomage              p=0.2345  normale ✓
  taux_court           p=0.1539  normale ✓
  taux_long            p=0.1348  normale ✓
  taux_change          p=0.7339  normale ✓

MATRICE DE CORRÉLATION :
             pib_growth  inflation  chomage  taux_court  taux_long  \
pib_growth        1.000     -0.065   -0.288       0.070      0.301   
inflation        -0.065      1.000   -0.285      -0.197     -0.244   
chomage          -0.288     -0.285    1.000      -0.018     -0.171   
taux_court        0.070     -0.197   -0.018       1.000      0.686   
taux_long         0.301     -0.244   -0.171       0.686      1.000   
taux_change       0.085      0.616   -0.247      -0.046     -0.083   

             taux_change  
pib_growth         0.085  
inflation          0.616  
chomage           -0.247  
taux_court        -0

In [17]:
# =============================================================================
# ÉTAPE 5 : ANALYSES AVANCÉES
# =============================================================================
print("\n[5/6] Analyses avancées (sensibilité, reverse stress, validation)...")
t0 = time.time()

from advanced_analysis import (sensitivity_analysis_v2,
                                       reverse_stress_test,
                                       validate_model,
                                       plot_advanced_analysis)

sensitivity   = sensitivity_analysis_v2(df, model, scaler, feature_names)
critical_lvls = reverse_stress_test(df, model, scaler, feature_names, lcr_threshold=80.0)
validation    = validate_model(df, X, y, feature_names)

# Sauvegarde JSON de la validation
val_json = {k: float(v) if isinstance(v, (np.floating, float)) else
            (bool(v) if isinstance(v, (np.bool_, bool)) else
             [float(x) for x in v] if isinstance(v, list) else v)
            for k, v in validation.items()}
with open(f'{OUTPUT_DIR}/model_validation.json', 'w') as f:
    json.dump(val_json, f, indent=2)

fig_adv = plot_advanced_analysis(sensitivity, critical_lvls, validation,
                                  save_path=f'{OUTPUT_DIR}/05_advanced_analysis.png')
plt.close()
print(f"  ✓ Temps : {time.time()-t0:.1f}s")


[5/6] Analyses avancées (sensibilité, reverse stress, validation)...

ANALYSE DE SENSIBILITÉ (ceteris paribus)
------------------------------------------------------------
  Flux de référence (médianes) : 32.456 Mds XOF

  pib_growth                élasticité = +0.0126  impact 1σ = +0.169 Mds XOF
  inflation                 élasticité = -0.0007  impact 1σ = +0.001 Mds XOF
  chomage                   élasticité = -0.0267  impact 1σ = -0.120 Mds XOF
  taux_court                élasticité = +0.0035  impact 1σ = +0.032 Mds XOF
  taux_long                 élasticité = -0.0175  impact 1σ = -0.137 Mds XOF

REVERSE STRESS TESTING — Seuil : LCR = 80.0%
  HQLA disponible    : 176.6 Mds XOF
  Flux critique max  : 220.8 Mds XOF
  (Au-delà de ce niveau de flux, LCR < 80.0%)

  pib_growth           seuil non atteint dans [-6σ, +6σ]
  inflation            seuil non atteint dans [-6σ, +6σ]
  chomage              seuil non atteint dans [-6σ, +6σ]
  taux_court           seuil non atteint dans [-6σ, +6σ

In [18]:
# =============================================================================
# ÉTAPE 6 : RAPPORT DE SYNTHÈSE
# =============================================================================
print("\n[6/6] Génération du rapport de synthèse...")

report_lines = []
report_lines.append("="*70)
report_lines.append("RAPPORT DE STRESS TEST DE LIQUIDITÉ BANCAIRE")
report_lines.append("Conforme Bâle III / BCEAO / EBA")
report_lines.append("="*70)

report_lines.append(f"\nDATE D'EXÉCUTION : {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}")
report_lines.append(f"MODÈLE ML UTILISÉ : {model_name}")
report_lines.append(f"VALIDITÉ DU MODÈLE : R²={validation['mean_r2']:.4f}  RMSE={validation['mean_rmse']:.3f} Mds XOF")
report_lines.append(f"NOMBRE DE SIMULATIONS : 10 000 par scénario")

report_lines.append("\n" + "="*70)
report_lines.append("RÉSULTATS PAR SCÉNARIO")
report_lines.append("="*70)

for _, row in risk_measures.iterrows():
    report_lines.append(f"\n{row['Scénario'].upper()} :")
    report_lines.append(f"  LCR médian    : {row['LCR médian (%)']:.1f}%")
    report_lines.append(f"  VaR 99%       : {row['VaR 99% (Mds)']:.2f} Mds XOF")
    report_lines.append(f"  ES 99%        : {row['ES 99% (Mds)']:.2f} Mds XOF")
    report_lines.append(f"  P(LCR < 100%) : {row['P(LCR<100%) %']:.2f}%")

report_lines.append("\n" + "="*70)
report_lines.append("FICHIERS PRODUITS")
report_lines.append("="*70)

files_produced = [
    'data_bancaire.csv         — Dataset bancaire (60 obs × variables)',
    '01_data_overview.png      — Aperçu données historiques',
    '02_ml_results.png         — Résultats et validation des modèles ML',
    '03_scenarios.png          — Hypothèses macroéconomiques par scénario',
    '04_monte_carlo.png        — Dashboard Monte-Carlo complet',
    '05_advanced_analysis.png  — Sensibilité, Reverse Stress, Validation',
    'scenario_results.csv      — Résultats bilan par scénario',
    'risk_measures.csv         — Mesures de risque (VaR, ES, P(breach))',
    'model_validation.json     — Métriques de validation technique',
    'rapport_risques.txt       — Commentaire économique automatique',
]

for f in files_produced:
    report_lines.append(f"  {f}")

full_report = "\n".join(report_lines)
print(full_report)

with open(f'{OUTPUT_DIR}/rapport_synthese.txt', 'w', encoding='utf-8') as f:
    f.write(full_report)

print("\n" + "="*70)
print("  PIPELINE COMPLET EXÉCUTÉ AVEC SUCCÈS")
print(f"  Sorties disponibles dans : {OUTPUT_DIR}/")
print("="*70)


[6/6] Génération du rapport de synthèse...
RAPPORT DE STRESS TEST DE LIQUIDITÉ BANCAIRE
Conforme Bâle III / BCEAO / EBA

DATE D'EXÉCUTION : 2026-06-14 12:41
MODÈLE ML UTILISÉ : XGBoost
VALIDITÉ DU MODÈLE : R²=-3.1940  RMSE=2.506 Mds XOF
NOMBRE DE SIMULATIONS : 10 000 par scénario

RÉSULTATS PAR SCÉNARIO

NORMAL :
  LCR médian    : 500.0%
  VaR 99%       : 37.91 Mds XOF
  ES 99%        : 38.74 Mds XOF
  P(LCR < 100%) : 0.00%

ADVERSE :
  LCR médian    : 500.0%
  VaR 99%       : 37.55 Mds XOF
  ES 99%        : 38.32 Mds XOF
  P(LCR < 100%) : 0.00%

SEVERE :
  LCR médian    : 500.0%
  VaR 99%       : 37.11 Mds XOF
  ES 99%        : 38.05 Mds XOF
  P(LCR < 100%) : 0.00%

SYSTEMIC :
  LCR médian    : 486.8%
  VaR 99%       : 37.02 Mds XOF
  ES 99%        : 37.84 Mds XOF
  P(LCR < 100%) : 0.00%

FICHIERS PRODUITS
  data_bancaire.csv         — Dataset bancaire (60 obs × variables)
  01_data_overview.png      — Aperçu données historiques
  02_ml_results.png         — Résultats et validation d